<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">Appendix A - Reading the Query Profile</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Appendix - Reading the Query Profile
#### Duration: 20 minutes

**In a live class:** This is provided to you as bonus material. You can review this appendix after class.

<br></br>
A reference for reading the Databricks **Query Profile** ([AWS](https://docs.databricks.com/aws/en/sql/user/queries/query-profile/) | [Azure](https://learn.microsoft.com/azure/databricks/sql/user/queries/query-profile/) | [GCP](https://docs.databricks.com/gcp/en/sql/user/queries/query-profile/)). It's organized around the profile itself, its panels, metrics and views.

### How to open a Query Profile

From a notebook cell click **See performance** under the cell output, then open the **Query profile**.

## The Details

Opening a profile shows the **Details** panel: the query's identity, its timing and its input and output totals. 

This is the fastest read on whether a query is expensive and why.

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Details.png" alt="Query Profile Details panel for the Demo 02 bad query" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

What's on this panel, top to bottom:

* **ID** and the query text (`_bad_rows = _bad_df.count()`), with **See query text** for the full statement.

* **Query wall-clock duration: 7 s 196 ms.** This is what the user waited (scheduling, optimization, then execution).

* **Bytes read: 367.35 MB**, with a funnel badge reading **0%**. The badge is the fraction **pruned** ([AWS](https://docs.databricks.com/aws/en/tables/data-skipping/) | [Azure](https://learn.microsoft.com/azure/databricks/tables/data-skipping/) | [GCP](https://docs.databricks.com/gcp/en/tables/data-skipping/)).

* **Rows read: 75,015,615.** **Files read: 96**, also **0%** pruned. 

* Bytes, rows and files written are all 0 because a `count()` returns a single number.

* **Query Source:** the notebook and cell that launched it ("02 Demo … Run the bad query").

* **Aggregated task time:** Tasks total time 1.45 m, Tasks time in **Photon** ([AWS](https://docs.databricks.com/aws/en/compute/photon/) | [Azure](https://learn.microsoft.com/azure/databricks/compute/photon/) | [GCP](https://docs.databricks.com/gcp/en/compute/photon/)) 90%.

### The two clocks

* Wall-clock duration (**7.196 s**) is what the user waited. 

* Aggregated task time (**1.45 m**, about 87 s) is the sum of every task's time across all cores. 

* Task time is always at least wall-clock. Here it is about **12x** larger, which means roughly twelve cores worked in parallel. 

* Much larger than wall-clock is healthy parallelism. 

* Close to wall-clock means the work funnelled through a few tasks or back to the driver.

### IO

Expanding the same panel shows the complete IO block:

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Aggregated Task Time in Photon Full.png" alt="Query Profile aggregated task time and IO breakdown" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

* **Rows returned 1** (the count), Rows read 75,015,615, Bytes read 367.35 MB.

* **Bytes pruned 0** and **Files pruned 0**: nothing was skipped, every one of the 96 files was opened.

* The disk cache percent is visible right here in the profile. **Bytes read from cache 72%**: 72% of the bytes were served from the local SSD **disk cache** ([AWS](https://docs.databricks.com/aws/en/optimizations/disk-cache/) | [Azure](https://learn.microsoft.com/azure/databricks/optimizations/disk-cache/) | [GCP](https://docs.databricks.com/gcp/en/optimizations/disk-cache/)) rather than remote storage. 

* **Partitions read 0** ([AWS](https://docs.databricks.com/aws/en/tables/partitions/) | [Azure](https://learn.microsoft.com/azure/databricks/tables/partitions/) | [GCP](https://docs.databricks.com/gcp/en/tables/partitions/)): the table is not partitioned.

* **Bytes spilled to disk 0** ([AWS](https://docs.databricks.com/aws/en/compute/sql-warehouse/warehouse-behavior/) | [Azure](https://learn.microsoft.com/azure/databricks/compute/sql-warehouse/warehouse-behavior/) | [GCP](https://docs.databricks.com/gcp/en/compute/sql-warehouse/warehouse-behavior/)): no spill.

**Where this varies later in the course:**

* **Pruning (0% here):** in Demo 04 the **liquid-clustered** ([AWS](https://docs.databricks.com/aws/en/tables/clustering/) | [Azure](https://learn.microsoft.com/azure/databricks/tables/clustering/) | [GCP](https://docs.databricks.com/gcp/en/tables/clustering/)) table prunes, so **Bytes read** drops sharply and this field rises above 0.

* **Bytes read from cache:** Demo 05 (caching) is about this disk-cache line and the separate result cache. A **result cache** ([AWS](https://docs.databricks.com/aws/en/sql/user/queries/query-caching/) | [Azure](https://learn.microsoft.com/azure/databricks/sql/user/queries/query-caching/) | [GCP](https://docs.databricks.com/gcp/en/sql/user/queries/query-caching/)) hit produces no profile at all ("Query profile is not available").

* **Bytes spilled to disk (0 here):** stays 0 on serverless, which sizes memory to the data. On classic compute or at production volumes it goes above 0 (Demo 07, spill).

* **Tasks time in Photon (90% here):** the Demo 08 UDF query drops this to 5% (see the Photon section below).

## Operators

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Top Operators Selected.png" alt="Query Profile Top Operators panel and operator graph, colored by time spent" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

### The operator graph (right side)

**The graph is the query's physical plan**, read **bottom to top**: 

**Four Scan Table** ([AWS](https://docs.databricks.com/aws/en/sql/user/queries/query-profile/) | [Azure](https://learn.microsoft.com/azure/databricks/sql/user/queries/query-profile/) | [GCP](https://docs.databricks.com/gcp/en/sql/user/queries/query-profile/)) nodes at the bottom (one per table: `order_items`, `orders`, `customers`, `products`) feed **Shuffle** ([AWS](https://docs.databricks.com/aws/en/optimizations/aqe/) | [Azure](https://learn.microsoft.com/azure/databricks/optimizations/aqe/) | [GCP](https://docs.databricks.com/gcp/en/optimizations/aqe/)) and **Inner Join** nodes. 

Three **Inner Join** operators chain the four tables together and **Grouping Aggregate** ([AWS](https://docs.databricks.com/aws/en/transform/aggregation/) | [Azure](https://learn.microsoft.com/azure/databricks/transform/aggregation/) | [GCP](https://docs.databricks.com/gcp/en/transform/aggregation/)) nodes collapse rows up to the **Result Query Stage** at the top. 

Each node is numbered (#1 to #23) and labeled with its specific operator. 

Edges are labeled with the **rows** flowing between operators, so you can watch the volume change: 
* the `order_items` scan emits **60 M rows**
* the three joins reduce through **30.09 M**, **12.88 M**, and **4 M**
* then collapse to **1** row at the top (the count). 

Zoom controls and the minimap are bottom-right.

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Top Operators.png" alt="Query Profile Top Operators" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

### The Top operators panel (left side)

* **Aggregated operators** groups every operator by type with its share of time: **Shuffle 49.3%**, **Grouping Aggregate 19.0%**, **Scan 18.2%**, **Join 13.5%**, **Aggregate 0%**.

* **Shuffle**, which is data movement, is about half the query.

* The ranked list below names the single costliest operators: **Shuffle #10 on `CAST(oi.product_idx AS BIGINT)`, 13.48 s**, then **Grouping Aggregate #8, 10.15 s**, **Shuffle #20, 7.74 s**, **Inner Join #12, 5.95 s**, and so on

* The darkest node in the graph, **Shuffle #10**, is the single hottest operator.

### Selecting an operator type

Clicking an operator type filters the list to it and highlights every instance in the graph. 

Below **Shuffle** is selected so all **8** shuffle nodes light up and the list shows only shuffles.

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Top Operators Selected.png" alt="Query Profile with the Shuffle operator type selected, highlighting all shuffle nodes" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

**Where this varies later:** Demo 07 dives deeper into shuffle and join shapes. The join here is a **shuffle hash join** ([AWS](https://docs.databricks.com/aws/en/transform/join/) | [Azure](https://learn.microsoft.com/azure/databricks/transform/join/) | [GCP](https://docs.databricks.com/gcp/en/transform/join/)) (both sides shuffled). Demo 07 contrasts it with a broadcast join, where the small side has no shuffle.

## Reading the graph by color

### Rows

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Rows.png" alt="Operator graph colored by rows" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

Colored by **Rows, 0 to 60 M**. The darkest nodes are the `order_items` **Scan Table #21** and its **Shuffle #20**, both at **60 M rows**. Follow the edges up and the row count falls: 60 M into the join, **30.09 M** out, then 12.88 M, 4 M, 25, and finally 1. 

This view makes the row volume through each operator visible at a glance.

### Memory peak

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Memory Peak.png" alt="Operator graph colored by peak memory" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

Colored by **Memory (GB), 0 to 5.53 GB**. The darkest nodes are the **Shuffle** operators (the heaviest peaks near **5.53 GB**), while scans and joins are light.

**Where this varies later:** This is the leading indicator for spill, which for this query profile stays at 0. On classic compute those multi-GB shuffle buffers are where spill to disk would appear.

## Photon and UDFs

The query above ran **90%** in **Photon** ([AWS](https://docs.databricks.com/aws/en/compute/photon/) | [Azure](https://learn.microsoft.com/azure/databricks/compute/photon/) | [GCP](https://docs.databricks.com/gcp/en/compute/photon/)). 

To see what leaving Photon looks like, here is **Demo 08's UDF query**:

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Arrow Eval in Python.png" alt="Arrow Eval Python" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

In the Top Operators panel for the UDF query, one operator trumps everything: **Arrow Eval Python** is **94.7%** of aggregated time and **11.92 m** on its own. 

The **Arrow Batch Source** (0.4%) and **Arrow Conversion** (0.2%) nodes are the Arrow serialization that moves data in and out of the Python worker. 

The real Scan, Shuffle and Join are extremely small compared to the UDF.

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/QP/QP - Aggregated Task Time in Photon.png" alt="Aggregated task time 12.51 minutes with 5 percent in Photon for the UDF query" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:24px 0px 12px 0px;" />

Only **5%** in Photon, compared to Demo 02's **90%**. 

The **Python UDF** ([AWS](https://docs.databricks.com/aws/en/udf/pandas/) | [Azure](https://learn.microsoft.com/azure/databricks/udf/pandas/) | [GCP](https://docs.databricks.com/gcp/en/udf/pandas/)) pushed almost all of the work off Photon and into a Python worker. 

The `Arrow Eval Python` operator plus a low **Tasks time in Photon** is how you spot a UDF cost in the profile.

## Lab 1: Reading the Scan Table Operator

In Lab 1 you inherit a report that filters on `account_key` but the table is partitioned by `event_date`. 

The Query Profile's **Scan Table** operator is where the mismatch becomes visible. 

Select the Scan Table node in the operator graph to open its detail panel.

### Before: partitioned by `event_date`, filtering on `account_key`

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/lab/Lab 1 - Before Report.png" alt="Scan Table operator for the before query, 0 percent pruned" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

What the Scan Table detail panel shows:

* **Files read: 730**, one per daily partition. The table was partitioned by `event_date` across two years, producing one tiny file per day.

* **Pruning badge: 0%.** No files were skipped. The Details panel at the top confirms this same number as **Files pruned: 0**.

* **Bytes read** covers the entire table (184 MB), even though the report needs only one account's rows.

* **Partition filter applied: none visible** for `account_key`. The only partition column is `event_date`, which doesn't appear in the report's WHERE clause

* **Dynamic filters / data skipping:** each daily file contains rows for many accounts, so every file's min/max range for `account_key` spans the filter value. Data skipping (file-level min/max pruning) cannot eliminate any file either.

This profile answers all three diagnostic questions from the lab:

1. **How is the table laid out?** Partitioned by `event_date`, 730 files, about 252 KB per file, far below the 1 GB minimum partition size guideline. This is over-partitioning.

2. **Does the layout match the filter?** No. The report filters on `account_key` but the table is partitioned on `event_date`. Partition pruning only fires when you filter on the partition column, so it does nothing here.

3. **Why is nothing pruned?** A single account is active across many days. Its rows are scattered across every daily file. Each file's min and max for `account_key` spans the filter value, so data skipping cannot eliminate a single file. The result: 730 files read, 0 pruned.

---

### After: clustered by `account_key`, same filter

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/lab/Lab 1 - After Report.png" alt="Scan Table operator for the after query with high pruning" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

After recreating the table with `CLUSTER BY (account_key)`, the same report produces a radically different Scan Table panel:

* **Files read** drops from 730 to a small number (typically 1 to 3). Liquid clustering co-located all rows for a single `account_key` into a compact region of the table.

* **Pruning badge: 90%+.** The vast majority of files were skipped. Their min/max stats for `account_key` don't contain the filter value, so data skipping eliminates them before any bytes are read.

* **Bytes read** drops from 184 MB to a few MB, because the scan opens only the files that contain the target account.

* **Rows read** drops proportionally. Instead of scanning 75 M+ rows across the whole table, the scan touches only the rows belonging to that one account.

* **The operator time** for Scan Table shrinks accordingly and since the scan was the dominant cost, wall-clock duration drops as well.

### Why the change works

Liquid clustering (`CLUSTER BY (account_key)`) physically re-arranges the data so that rows sharing the same `account_key` sit together in the same files. 

When the report filters on one account, the file-level min/max statistics now cleanly exclude most files. 

This is the transition from "scattered rows, every file opened" to "co-located rows, most files pruned."

The Scan Table operator is where you confirm this: compare **Files read** and the **pruning badge** between the two profiles. 

If pruning stays at 0% after clustering, either the clustering column doesn't match the filter or `OPTIMIZE` hasn't yet run.

## Lab 2: Python UDFs in the Batch Eval Python Operator Attributes

In Lab 2 you find the most expensive job and run it. The Query Profile's operator attributes are where the Python UDF becomes visible.

Select the operator that holds most of the task time and look at its attributes panel.

<img src="https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/lab/Lab 2 - Python UDF 0.png" alt="Operator attributes showing pythonUDF0 next to region and amount" style="max-width:100%;border-radius:6px;border:1px solid #e2e8ee;margin:12px 0;" />

What the attributes show:

* **`pythonUDF0`** appears in the operator's output list, next to `region` and `amount`. This is Spark's internal name for the first Python UDF expression in the query plan. It maps to the `settlement_tier_bad(region, amount)` call in the SQL.

* The presence of `pythonUDF0` in the attributes confirms that this operator is evaluating a Python UDF. Spark serializes the `region` and `amount` columns out to a Python worker, evaluates the function row by row, and serializes the result back.

* This is the direct evidence for Lab 2's diagnostic questions: the expensive operator is a Python UDF, it runs outside Photon, and it forces the Photon fallback that burns the task time.

### What to look for in the rest of the profile

* **Tasks time in Photon** in the Details panel will be low (around 1-5%), because the UDF operator dominates the aggregated task time and it runs entirely outside Photon.

* **Arrow Eval Python** or a similar Python-evaluation operator will appear in the Top Operators panel as the largest share of time.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>